<a href="https://colab.research.google.com/github/comcacanhca/RAG_BatTu/blob/main/Battu_RAG_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Cài đặt các thư viện cần thiết

In [ ]:
# Cài đặt các thư viện cần thiết và bộ xử lý tài liệu Word (.docx)
!pip install -q langchain langchain-huggingface
!pip install -U langchain-text-splitters langchain-unstructured
!pip install -q chromadb rank_bm25 sentence-transformers docx2txt # Allow pip to install the latest compatible chromadb
!pip install langchain_community langchain_experimental
!pip install python-docx markdownify langchain-core
!pip install gptqmodel


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.6/169.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 74.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.

### 2. Import thư viện và kết nối Google Drive
Hãy chạy cell dưới đây để mount Google Drive của bạn. Đường dẫn thư mục tài liệu sẽ được thiết lập tự động.

In [ ]:
import os
from google.colab import drive
import glob
# Mount Google Drive
drive.mount('/content/drive')

# Đường dẫn đến folder tài liệu của bạn (thay đổi nếu cần)
# Folder ID của bạn: 1tBYxOGTaeS9CaAbWOXDw2hVz0WadP4JJ
# Sau khi mount, bạn có thể tìm thư mục này trong "Shared with me" hoặc "My Drive" tùy cách bạn thêm phím tắt.
DOCS_DIR = "/content/drive/MyDrive/RAG_Documents"
docx_files = glob.glob(os.path.join(DOCS_DIR + '/', "*.docx"), recursive=True)

for i in docx_files:#glob.glob(os.path.join(DOCS_DIR + "/",'*')):
    print(f'Found doc: {i}')
print(f"Thư mục tài liệu của bạn đã được cấu hình tại: {DOCS_DIR}")
print("Hãy đảm bảo bạn đã tải các tài liệu cần thiết vào thư mục này.")

Mounted at /content/drive
Found doc: /content/drive/MyDrive/RAG_Documents/Manh phái sơ , Trung, Cao cấp mệnh lý học.docx
Thư mục tài liệu của bạn đã được cấu hình tại: /content/drive/MyDrive/RAG_Documents
Hãy đảm bảo bạn đã tải các tài liệu cần thiết vào thư mục này.


### 3. Xây dựng cấu trúc Parent-Child Semantic Chunking & Hybrid Retriever với Reranker

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_core.documents import Document
import glob
import os
import re
import numpy as np
from typing import List, Dict, Any, Tuple
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, util

# --- DANH SÁCH TỪ KHÓA CHUYÊN NGÀNH ĐỂ TRÁNH BỊ CHUNKING/TOKENIZE SAI ---
DOMAIN_KEYWORDS = [
    "nhật can", "nhật chi", "nhật chủ", "nguyệt lệnh", "dụng thần", "hỷ thần", "kỵ thần",
    "tòng cách", "bát cách", "giả tòng cách", "lộc thần", "tỷ kiếp", "thương quan",
    "thực thần", "kiếp tài", "chính quan", "chính tài", "thiên tài", "thiên quan",
    "tố công", "công thần", "thành đảng", "thành thế", "mộc hỏa nhất đảng", "kim thủy nhất đảng",
    "táo thổ", "thấp thổ", "thìn tuất xung", "sửu mùi xung", "tị thân hợp", "mão thìn hại"
]

# --- GIAI ĐOẠN 1 & 2: CUSTOM TOKENIZER & QUERY TRANSFORMER ---
class DomainBM25Tokenizer:
    def __init__(self, domain_keywords: List[str]):
        self.keywords = sorted(domain_keywords, key=len, reverse=True)
        pattern = "|".join(re.escape(k) for k in self.keywords)
        self.regex = re.compile(f"({pattern})", re.IGNORECASE)

    def tokenize(self, text: str) -> List[str]:
        def replace_func(match):
            return match.group(0).replace(" ", "_").lower()
        text_processed = self.regex.sub(replace_func, text)
        return re.findall(r'\b\w+\b', text_processed.lower())

class QueryTransformer:
    def __init__(self, llm_pipeline=None):
        self.llm_pipeline = llm_pipeline

    def transform(self, user_query: str) -> Dict[str, str]:
        if not self.llm_pipeline:
            expanded_query = f"{user_query} Bát tự cách cục dụng thần hỷ thần"
            hyde_doc = f"Luận về {user_query}: Cần xét căn gốc nhật chủ, nguyệt lệnh, tòng cách hoặc bát cách để định dụng thần."
            return {"expanded_query": expanded_query, "hyde_doc": hyde_doc}

        prompt = f"<start_of_turn>user\nHãy viết một câu giải thích học thuật ngắn gọn theo chuẩn thuật ngữ Bát Tự cho khái niệm: '{user_query}'\n<end_of_turn>\n<start_of_turn>model\n"
        try:
            hyde_doc = self.llm_pipeline(prompt)[0]['generated_text'].strip()
        except Exception:
            hyde_doc = user_query
        return {"expanded_query": user_query, "hyde_doc": hyde_doc}

# --- CẤU HÌNH THIẾT BỊ EMBEDDING & RERANKER AN TOÀN ---
print("Đang tải Embedding Model với cấu hình Batch Size tối ưu...")
embeddings = HuggingFaceEmbeddings(
    model_name="keepitreal/vietnamese-sbert",
    encode_kwargs={"batch_size": 128}
)

CHROMA_DB_DIR = "/content/chroma_db"
print("Đang tải Reranker Model...")
reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

def load_documents_from_directory(docs_directory):
    pdf_files = glob.glob(os.path.join(docs_directory + '/', "*.pdf"), recursive=True)
    docx_files = glob.glob(os.path.join(docs_directory + '/', "*.docx"), recursive=True)
    print(f'Found: {len(docx_files)} docx')
    all_docs = []
    for file in docx_files:
        try:
            loader = Docx2txtLoader(file)
            all_docs.extend(loader.load())
        except Exception as e:
            print(f"Lỗi khi đọc file Word {file}: {e}")
    return all_docs

/tmp/ipykernel_2536/1540593757.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
/tmp/ipykernel_2536/1540593757.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Đang tải Embedding Model với cấu hình Batch Size tối ưu...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

# 4. Tích hợp Gemma Model và thiết lập Pipeline RAG (với tùy chọn Reasonilng)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from huggingface_hub import login

# 1. HỖ TRỢ ĐĂNG NHẬP HOẶC TỰ ĐỘNG CHUYỂN SANG MODEL KHÔNG GATED
print("=== THIẾT LẬP MÔ HÌNH NGÔN NGỮ ===")

# Qwen2.5-1.3B-Instruct là model cực mạnh cho tiếng Việt, không bị khóa (non-gated)
model_id = "Qwen/Qwen2.5-3B-Instruct-AWQ"
print(f"Không dùng Token. Hệ thống tự động chuyển sang mô hình: {model_id}")

print(f"Đang tải mô hình {model_id}...")
# 1. Tải Tokenizer và cấu hình Pad Token
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 2. Tải Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype='auto',
    trust_remote_code=True
)


# 3. Khởi tạo Pipeline đã tối ưu tham số cho RAG
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,           # Đủ dài cho câu trả lời RAG thông thường (có thể hạ xuống 256 nếu cần phản hồi cực nhanh)
    do_sample=True,                # Bắt buộc bật do_sample khi tùy chỉnh temperature / top_p
    temperature=0.1,               # Hạ thấp xuống 0.1 để giảm tối đa ảo giác (Hallucination), giúp LLM bám sát Context
    top_p=0.9,
    repetition_penalty=1.1,        # Tránh việc LLM bị lặp lại từ ngữ trong context
    return_full_text=False,        # QUAN TRỌNG: Chỉ trả về câu trả lời mới, KHÔNG lặp lại prompt/context
    pad_token_id=tokenizer.pad_token_id
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

#5 Tạo vector store


In [ ]:
import gc
import torch
import chromadb
from langchain_community.vectorstores import Chroma
from concurrent.futures import ThreadPoolExecutor
import numpy as np

# --- GIAI ĐOẠN 3: MIN-MAX RELATIVE SCORE FUSION ---
class RelativeScoreFusion:
    @staticmethod
    def _min_max_scale(scores_dict: Dict[str, float]) -> Dict[str, float]:
        if not scores_dict:
            return {}
        vals = list(scores_dict.values())
        min_v, max_v = min(vals), max(vals)
        if max_v == min_v:
            return {k: 1.0 for k in scores_dict}
        return {k: (v - min_v) / (max_v - min_v) for k, v in scores_dict.items()}

    def fuse(self, sparse_results: List[Tuple[str, float]], dense_results: List[Tuple[str, float]], alpha: float = 0.6) -> List[Tuple[str, float]]:
        sparse_norm = self._min_max_scale(dict(sparse_results))
        dense_norm = self._min_max_scale(dict(dense_results))
        all_ids = set(sparse_norm.keys()).union(set(dense_norm.keys()))

        fused = {}
        for doc_id in all_ids:
            fused[doc_id] = alpha * sparse_norm.get(doc_id, 0.0) + (1.0 - alpha) * dense_norm.get(doc_id, 0.0)
        return sorted(fused.items(), key=lambda x: x[1], reverse=True)

# --- COMPATIBLE RETRIEVER WRAPPER CHO PIPELINE HIỆN TẠI ---
class DomainHybridRetriever:
    def __init__(self, parent_map, final_child_docs, embeddings, reranker_model):
        self.parent_map = parent_map
        self.child_docs = final_child_docs
        self.tokenizer = DomainBM25Tokenizer(DOMAIN_KEYWORDS)
        self.fuser = RelativeScoreFusion()

        # Sparse Engine
        corpus_tokens = [self.tokenizer.tokenize(d.page_content) for d in final_child_docs]
        self.bm25 = BM25Okapi(corpus_tokens)

        # Dense Engine (Chroma)
        self.vectorstore = Chroma.from_documents(documents=final_child_docs, embedding=embeddings)
        self.reranker = reranker_model

    def invoke(self, query: str, top_k: int = 30) -> List[Document]:
        # 1. Sparse Search
        tokens = self.tokenizer.tokenize(query)
        bm25_scores = self.bm25.get_scores(tokens)
        top_sparse_idx = np.argsort(bm25_scores)[::-1][:30]
        sparse_res = [(f"doc_{i}", float(bm25_scores[i])) for i in top_sparse_idx if bm25_scores[i] > 0]

        # 2. Dense Search
        dense_docs = self.vectorstore.similarity_search_with_relevance_scores(query, k=30)
        dense_res = [(f"doc_{self.child_docs.index(doc)}", float(score)) for doc, score in dense_docs if doc in self.child_docs]

        # 3. Fuse Scores
        fused_res = self.fuser.fuse(sparse_res, dense_res, alpha=0.4)[:top_k]

        # 4. Map back to LangChain Documents and Rerank
        candidate_docs = []
        for doc_id, _ in fused_res:
            idx = int(doc_id.split("_")[1])
            candidate_docs.append(self.child_docs[idx])

        if not candidate_docs:
            return []

        pairs = [(query, doc.page_content) for doc in candidate_docs]
        scores = self.reranker.score(pairs)
        reranked = sorted(zip(candidate_docs, scores), key=lambda x: x[1], reverse=True)

        return [doc for doc, score in reranked[:3]]

def prepare_rag_retriever(docs_directory):
    print(f"Đang tải tài liệu gốc từ... {docs_directory}")
    all_docs = load_documents_from_directory(docs_directory)
    if not all_docs:
        all_docs = [Document(page_content="Tài liệu mẫu: Chương trình RAG Bát Tự.", metadata={"source": "sample.txt"})]

    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100,
        separators=["\n\n", "Chương ", "\n", ". ", " ", ""],
                                                     )
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=350,
        chunk_overlap=60,
        separators=["\n\n", "\nVí dụ", "\nCàn tạo:", "\nKhôn tạo:", "\n", ". ", " ", ""],
        strip_whitespace=True
    )
    parent_docs = parent_splitter.split_documents(all_docs)
    final_child_docs = []
    parent_map = {}

    for idx, p_doc in enumerate(parent_docs):
        parent_id = f"parent_{idx}"
        p_doc.metadata["parent_id"] = parent_id
        parent_map[parent_id] = p_doc.page_content

        chunks = child_splitter.split_documents([p_doc])
        for chunk in chunks:
            chunk.metadata["parent_id"] = parent_id
            chunk.metadata["parent_content"] = p_doc.page_content
            final_child_docs.append(chunk)

    # Khởi tạo Pipeline nâng cao mới tích hợp trong Retriever
    hybrid_retriever = DomainHybridRetriever(parent_map, final_child_docs, embeddings, reranker_model)
    return hybrid_retriever, parent_docs, final_child_docs, parent_map

retriever, parent_documents, child_docs, parent_map = prepare_rag_retriever(DOCS_DIR)
print("Hệ thống Domain-Specific Hybrid Retriever đã thiết lập thành công!")

Đang tải tài liệu gốc từ... /content/drive/MyDrive/RAG_Documents
Found: 1 docx


### 6. Kiểm tra thủ công Hybrid Search




In [ ]:
eval_questions = [
  {
    "question": "Theo Manh Phái, khi xem xét một Bát Tự thì vị trí nào cần được ưu tiên xem xét đầu tiên để biết bát tự công ở đâu?",
    "ground_truth_answer": "Đầu tiên phải xem Nhật can, thứ hai xem Nhật chi để xác định bát tự công ở nơi nào."
  },
  {
    "question": "Hai điểm quan trọng nhất cần chú trọng khi xem xét Nhật can là gì?",
    "ground_truth_answer": "Thứ nhất xem Nhật can có hợp hay không (hợp chính quan hoặc chính tài). Thứ hai, nếu không hợp, xem Nhật can có sinh hay không (có thực thần, thương quan gần sát hay không)."
  },
  {
    "question": "Nếu Nhật can không có hợp cũng không có sinh thì phương pháp xem mệnh tiếp theo là gì?",
    "ground_truth_answer": "Sẽ vứt bỏ Nhật can không xem, chuyên xem Nhật chi cùng các chi khác xem có thành đảng thành thế hay không, hoặc phát sinh các quan hệ hình, xung, khắc, hại, mộ, tam hợp, lục hợp, ám hợp."
  },
  {
    "question": "Khi nào thì việc tố công bằng Lộc thần hoặc Tỷ kiếp khiến cho cấp độ Bát Tự không cao?",
    "ground_truth_answer": "Khi chỉ có Lộc thần hoặc Tỷ kiếp đến tố công, vì Lộc chủ về thân thể, thường là người lao động chân tay chân chính, không có thành tựu đặc biệt lớn."
  },
  {
    "question": "Làm thế nào để xác định chủ thể chính tham gia tố công giữa Nhật can, Nhật chi và Nhật trụ?",
    "ground_truth_answer": "Nếu Nhật can tham dự tố công thì lấy Nhật can làm chủ; nếu Nhật can không làm công mà Nhật chi làm công thì lấy Nhật chi làm chủ; nếu cả can và chi cùng tham dự và nhất trí ý tứ thì lấy Nhật trụ làm chủ."
  },
  {
    "question": "Quan điểm của Manh Phái khác biệt thế nào với mệnh lý truyền thống về tiêu chuẩn phú quý bần tiện?",
    "ground_truth_answer": "Manh Phái không lấy sự thăng bằng của Bát Tự làm tiêu chuẩn, mà cho rằng phàm là bát tự đại phú quý nhất định phải bất thăng bằng, tức là thiên lệch về một phương 'Thế' ở chủ vị để làm công."
  },
  {
    "question": "Thế nào được gọi là 'Thế' hay 'Đảng' trong một Bát Tự phú quý theo Manh Phái?",
    "ground_truth_answer": "Thế chính là đảng, thể hiện sự kết hợp của một nhóm thần sát hoặc ngũ hành đồng lòng hợp lực để hoàn thành một việc làm công lớn, giống như có đoàn đội hỗ trợ."
  },
  {
    "question": "Phân biệt vai trò của chính công thần và phụ trợ công thần qua ví dụ Tị hỏa chế Thân kim ngộ Mão mộc?",
    "ground_truth_answer": "Tị hỏa trực tiếp chế Thân kim là chính công thần. Mão mộc sinh trợ cho Tị hỏa là phụ trợ công thần."
  },
  {
    "question": "Hai điều kiện bắt buộc phải đồng thời xuất hiện để tạo nên một Bát Tự có đại tác phẩm là gì?",
    "ground_truth_answer": "Phải đầy đủ hai điểm: thứ nhất là phải có Thế (đảng), thứ hai là phải có Công (tác động chế ước). Chỉ có thế không có công hoặc ngược lại đều không thể làm nên đại nghiệp."
  },
  {
    "question": "Kể tên các loại đảng hay thế phối hợp phổ biến thường gặp trong thực tế luận mệnh?",
    "ground_truth_answer": "Mộc hỏa nhất đảng, kim thủy nhất đảng, thủy mộc nhất đảng, hỏa táo thổ nhất đảng, kim thấp thổ nhất đảng, thủy thấp thổ nhất đảng."
  },
  {
    "question": "Khả năng chế ước của Táo thổ thế đối với các ngũ hành khác như thế nào?",
    "ground_truth_answer": "Táo thổ thế (đất khô nóng) có thể phá Thủy, phá Kim, và phá cả Thấp thổ."
  },
  {
    "question": "Khả năng chế ước của Thấp thổ thế đối với các ngũ hành khác ra sao?",
    "ground_truth_answer": "Thấp thổ thế (đất ẩm) có thể phá Hỏa và phá Táo thổ."
  },
  {
    "question": "Trong quan hệ xung giữa Thìn và Tuất, thông thường chữ nào chế chữ nào và trường hợp ngoại lệ là gì?",
    "ground_truth_answer": "Thông thường Thìn chế Tuất (thủy chế hỏa). Tuy nhiên nếu hỏa hỏa táo thổ thành thế cực mạnh thì Tuất lại chế được Thìn."
  },
  {
    "question": "Thế nào là 'Trung thần' trong một cục diện kết đảng không đồng nhất với Nhật chủ?",
    "ground_truth_answer": "Trung thần là chữ cốt lõi, đại diện chủ yếu nhất trong đảng đứng ra thực hiện nhiệm vụ tác công chế ước đối phương."
  },
  {
    "question": "Trong ví dụ Bát tự của Bạc Nhất Ba (Mậu Giáp Tân Quý / Thân Dần Mão Tỵ), tại sao nói Dần mộc là chính công thần?",
    "ground_truth_answer": "Vì mộc hỏa thành đảng muốn dồn kim thủy, trong đó Dần Thân xung là Tài trực tiếp chế Kiếp tài nên Dần mộc chính là chính công thần tác công lớn nhất."
  },
  {
    "question": "Hiện tượng 'Phản hướng tố công' trong mệnh Bạc Nhất Ba được thể hiện qua chi tiết nào?",
    "ground_truth_answer": "Là việc chủ vị rơi xuống dụng (Tài thần), khách vị rơi xuống thể (Kiếp tài), dùng dụng ở chủ vị để chế thể ở khách vị."
  },
  {
    "question": "Trong ví dụ Bát tự chơi cổ phiếu (Mậu Kỷ Quý Kỷ / Thân Mùi Tỵ Mùi), tại sao Thân kim lại bị xem là Tài?",
    "ground_truth_answer": "Tỵ Thân hợp là Tài tinh chế Kiếp tài cùng Ấn thụ ở khách vị, do đó khi Kiếp tài bị chế hoàn toàn thì được xem như Tài tinh."
  },
  {
    "question": "Giải thích ý nghĩa quản lý của tượng Mậu Quý hợp trong mệnh của nhà cái chơi cổ phiếu?",
    "ground_truth_answer": "Mậu Quý hợp thể hiện ý nghĩa là sự quản lý, khống chế và thao túng dòng tiền lớn một cách chủ động."
  },
  {
    "question": "Tại sao nói trong Bát tự (Ất Bính Tân Mậu / Tỵ Tuất Hợi Tuất), Hợi thủy thương quan bị kẹp chế không hoàn toàn?",
    "ground_truth_answer": "Vì Hợi thủy bị kẹp ở giữa hai chi Tuất thổ kẹp khắc, trong phương thức tố công thì hình thức kẹp chế này luôn được coi là chế không sạch sẽ hoàn toàn."
  },
  {
    "question": "Nguyên lý 'Chế khử tài và tài nguyên thần' đem lại kết quả gì cho mệnh chủ ở đại vận hỷ thần?",
    "ground_truth_answer": "Đem lại kết quả là phát tài lớn lên tới hàng triệu, hàng chục triệu (phát cự tài) khi bước vào đúng tuế vận."
  },
  {
    "question": "Tại sao Bát tự (Nhâm Kỷ Quý Tân / Thân Dậu Tỵ Dậu) lại ngã và lụi bại khi bước vào vận Dần?",
    "ground_truth_answer": "Vì Dần vận xung mất Thân kim, hình phạt động Tỵ hỏa, làm phá hủy hoàn toàn chính công thần Thân kim của nguyên cục."
  },
  {
    "question": "Trong Bát tự (Ất Kỷ Nhâm Tân / Tỵ Sửu Thìn Sửu), tại sao một chữ Tị hỏa đơn độc lại bị chế sạch sẽ?",
    "ground_truth_answer": "Vì thấp thổ và kim thủy kết đảng, Tị hỏa sinh Sửu và Thìn, lại củng Sửu thấp thổ, toàn bộ năng lượng của Tị hỏa bị tiết sạch vào thấp thổ chủ vị."
  },
  {
    "question": "Khái niệm 'Tài thống quan' trong Manh phái được hiểu như thế nào?",
    "ground_truth_answer": "Là khi quan tinh nhiều mà tài tinh ít, lúc này toàn bộ quan tinh sẽ được xem và tính toán trực tiếp như tài tinh."
  },
  {
    "question": "Địa chi đóng vai trò gì quyết định năng lượng của nguyên cục trong phương pháp chế của Manh Phái?",
    "ground_truth_answer": "Địa chi đóng vai trò quyết định vì năng lượng thực tế tích tụ dưới địa chi lớn hơn thiên can rất nhiều."
  },
  {
    "question": "Trong ví dụ làm quan (Đinh Nhâm Canh Nhâm / Hợi Tý Thìn Ngọ), hành động xung nào đóng vai trò tác công chính?",
    "ground_truth_answer": "Hành động Tý thủy xung Ngọ hỏa là phương thức chính để kim thủy và thấp thổ đảng chế ngự hoàn toàn hỏa."
  },
  {
    "question": "Quy luật 'Chế khử quan và quan nguyên thần' tương ứng với thành tựu gì?",
    "ground_truth_answer": "Tương ứng với việc đạt được chức vị cao, quyền lực lớn trong bộ máy chính quyền (làm quan đại thần)."
  },
  {
    "question": "Mệnh Chu tổng lý (Mậu Giáp Đinh Giáp / Tuất Dần Mão Thìn) sử dụng quan hệ hại nào để làm công chính?",
    "ground_truth_answer": "Sử dụng quan hệ Mão Thìn hại, kết hợp Thìn Tuất xung để chế ngự hoàn toàn Thìn thổ (quan sát khố)."
  },
  {
    "question": "Trong mệnh phú quý (Đinh Quý Bính Nhâm / Mùi Sửu Tý Thìn), tại sao không có tài nhưng vẫn là ức vạn phú ông?",
    "ground_truth_answer": "Vì bát tự không có tài tinh hiển lộ thì lấy thực thương (Sửu thổ) làm tài để chế khố, chế dương nhận, tạo ra lượng tố công cực kỳ lớn."
  },
  {
    "question": "Ý nghĩa biểu thị của 'Ấn khố' trong mô hình liên kết xí nghiệp là gì?",
    "ground_truth_answer": "Ấn là đơn vị công tác, Ấn khố biểu thị cho sự liên kết của rất nhiều xí nghiệp, tương đương một tập đoàn tài chính lớn."
  },
  {
    "question": "Trong mệnh (Mậu Kỷ Canh Đinh / Thìn Mùi Ngọ Hợi), tại sao lại luận đoán là phát tài chứ không phải làm quan?",
    "ground_truth_answer": "Vì hỏa táo thổ kết đảng chế ngự thực thần Hợi thủy, Hợi thủy là tài và tài nguyên thần nên khi bị chế sạch sẽ chủ về cự phú đại tài."
  },
  {
    "question": "Quyết 'Tọa khố hỷ hình xung' áp dụng như thế nào trong ví dụ thăng quan phó tỉnh của mệnh Canh Tuất?",
    "ground_truth_answer": "Nhật chủ tọa Tuất là quan sát khố, khi hành vận Thìn đến xung khai Tuất thổ quan khố thì lập tức thăng tiến chức vụ tỉnh cấp."
  },
  {
    "question": "Tại sao trong Bát tự của Tưởng Giới Thạch, việc hành vận Thìn lại dẫn đến thất bại phải rút lui ra Đài Loan?",
    "ground_truth_answer": "Vì Thìn là thủy khố bảo vệ Hợi thủy khỏi bị chế, đồng thời Thìn xung phá Tuất thổ (công thần) làm phản cục thế hỏa táo thổ."
  },
  {
    "question": "Nếu Nhật chủ muốn tố công nhưng không có quan hệ Hợp hay Sinh thì phải tìm ở thần nào?",
    "ground_truth_answer": "Phải tìm xem có Lộc thần hoặc Tỷ kiếp ở các chi khác để thực hiện phương thức làm công hay không."
  },
  {
    "question": "Sự kết hợp giữa Kim và Thấp thổ tạo thành đảng có khả năng chế ngự loại thế nào?",
    "ground_truth_answer": "Đảng Kim và Thấp thổ có khả năng chế ngự, tiêu diệt thế lực của Mộc và Hỏa."
  },
  {
    "question": "Khi Thủy và Thấp thổ kết đảng, mục tiêu chế hướng tới ngũ hành nào?",
    "ground_truth_answer": "Thủy và Thấp thổ kết đảng hướng tới mục tiêu chế ngự Táo thổ hoặc Hỏa."
  },
  {
    "question": "Mối quan hệ 'mang tượng' của Mậu Thân trong mệnh Bạc Nhất Ba giúp gì cho việc đọc hiểu?",
    "ground_truth_answer": "Mậu là ấn ngồi trên Thân kiếp tài, Mậu thấu ra đại diện cho hỏa thổ đảng quản lý toàn bộ kiếp tài Thân kim thông qua Mậu Quý hợp."
  },
  {
    "question": "Tại sao phản hướng tố công lại coi trọng việc dụng chế thể hơn là thể chế dụng?",
    "ground_truth_answer": "Vì khi dụng ở chủ vị chế được thể ở khách vị, hiệu quả và giá trị của việc làm công đạt được năng lượng cực kỳ lớn và triệt để."
  },
  {
    "question": "Trong mệnh nhà cái (Mậu Kỷ Quý Kỷ / Thân Mùi Tỵ Mùi), tại sao vận Hợi lại bị phản cục dẫn đến ngồi tù?",
    "ground_truth_answer": "Hợi thủy kiếp tài xuất hiện trợ giúp cho Thân kim chống lại thế lực Hỏa Táo thổ, làm nghịch đảo hoàn toàn cơ chế Tỵ Thân hợp của nguyên cục."
  },
  {
    "question": "Hiện tượng 'Tặc thần và Bộ thần' hoạt động theo cơ chế nào trong tuế vận?",
    "ground_truth_answer": "Khi nguyên cục đã chế sạch kỵ thần (tặc thần), thì đến vận tặc thần xuất hiện sẽ giúp bộ thần (hỷ thần) lập công lớn thu giữ tặc thần."
  },
  {
    "question": "Trong mệnh (Ất Kỷ Nhâm Tân / Tỵ Sửu Thìn Sửu), tại sao Sửu thổ lại phải nhập vào Thìn mộ?",
    "ground_truth_answer": "Vì Thìn là tỷ kiếp khố tọa dưới ngày của Nhật chủ, Sửu thổ là đồng hành thấp thổ nên theo lý thuyết kết đảng sẽ tự động nhập vào Thìn mộ."
  },
  {
    "question": "Sự khác biệt giữa trạng thái Mộ và Khố của địa chi khi không có xung hình là gì?",
    "ground_truth_answer": "Địa chi chứa mộ nếu không được xung hình khai mở thì là mộ tử đóng kín, có xung hình khai mở mới thực sự trở thành khố chứa kho tàng."
  },
  {
    "question": "Tại sao việc Nhâm thủy tự hợp với Ngọ hỏa trong mệnh đại quan lại có thể chế được hỏa?",
    "ground_truth_answer": "Vì Nhâm thủy mang theo năng lượng của kim thủy đảng từ trên thiên can khắc chế trực tiếp xuống bản khí Đinh hỏa tàng trong Ngọ."
  },
  {
    "question": "Tại sao Thực thần chế quan sát khố trong mệnh (Đinh Nhâm Quý Bính / Mão Dần Mão Thìn) lại giúp đỗ đạt khoa bảng?",
    "ground_truth_answer": "Vì mộc thế mạnh mẽ từ Dần Mão dùng quan hệ Mão Thìn hại để chế ngự Thìn thổ quan sát khố một cách hữu hiệu, đắc công danh."
  },
  {
    "question": "Vì sao Tài tinh hư thấu trên can trong một số bát tự làm quan lại đại biểu cho tài hoa trí tuệ?",
    "ground_truth_answer": "Vì tài tinh không có căn rễ thì không biểu thị cho tiền bạc vật chất thực tế, mà chỉ là tượng của học vấn, tư duy và năng lực thuyết phục."
  },
  {
    "question": "Trong mệnh Tưởng Giới Thạch, tại sao Canh kim hư thấu trên thiên can lại bị coi là nhược điểm?",
    "ground_truth_answer": "Vì Canh kim hư thấu không bị hỏa của nguyên cục chế ngự trực tiếp, tạo ra kẽ hở khiến khí thế táo hỏa không thể bao trùm toàn vẹn."
  },
  {
    "question": "Lộc thần của Nhật chủ đại diện cho điều gì và tại sao lộc làm công thì cuộc đời vất vả?",
    "ground_truth_answer": "Lộc thần đại diện cho chính bản thân cơ thể vật lý của Nhật chủ. Khi dùng thân thể để làm công thì phải bôn ba lao lực như người lao động."
  },
  {
    "question": "Sự phối hợp giữa Mộc và Hỏa thành thế có năng lực phá hủy hành nào mạnh nhất?",
    "ground_truth_answer": "Mộc hỏa thành thế có năng lực khắc chế Thổ mạnh nhất và đồng thời có khả năng phá Kim đắc lực."
  },
  {
    "question": "Trong quan hệ tương hại giữa Mão và Thìn, ngũ hành nào bị tổn thương nhiều nhất?",
    "ground_truth_answer": "Mộc của Mão phá hủy cấu trúc Thổ của Thìn, làm tổn thương trực tiếp đến bản khí Thổ và thủy tàng trong Thìn khố."
  },
  {
    "question": "Tại sao việc tìm ra chính xác chủ thể làm công (công thần) lại là chìa khóa đọc hiểu Bát tự?",
    "ground_truth_answer": "Vì công thần là trung tâm vận hành của toàn cục, giúp ta biết rõ Nhật chủ dùng phương thức nào, đoàn đội nào để đạt được phú quý bần tiện."
  },
  {
    "question": "Đặc điểm năng lượng của giòn kim (như Mùi thổ đối với kim) có vai trò gì trong tố công?",
    "ground_truth_answer": "Mùi thổ là táo thổ có tính chất giòn kim (làm giòn, nóng chảy kim) nhưng lực khắc trực tiếp yếu hơn hỏa tinh, nên ít khi đóng vai trò chính công thần."
  }
]

# New section 6.1 Chạy check vector store

In [ ]:
import numpy as np
from sentence_transformers.util import cos_sim

# Tự động khởi tạo lại retriever nếu bị mất trạng thái
if 'retriever' not in globals():
    raise NameError("[Thông báo] Không tìm thấy 'retriever' trong bộ nhớ. Đang tự động xây dựng lại...")

print("=== BẮT ĐẦU ĐÁNH GIÁ TOÀN DIỆN HỆ THỐNG RETRIEVAL ===\n")

total_questions = len(eval_questions)
hits_at_1 = 0
hits_at_3 = 0
hits_at_30 = 0
reciprocal_ranks = []
precision_at_3_list = []
similarity_scores = []

for idx, item in enumerate(eval_questions):
    query = item["question"]
    gold_standard = item["ground_truth_answer"]

    print(f"[Đánh giá câu hỏi {idx+1}/{total_questions}]: \"{query}\"")

    # 1. Sử dụng trực tiếp truy vấn gốc
    search_query = query

    # 2. Truy xuất bằng retriever
    try:
        # Cấu hình lấy tối đa 30 tài liệu để đánh giá Hit Rate @ 30
        retrieved_docs = retriever.invoke(search_query, top_k=30)
    except Exception as e:
        print(f" -> Lỗi truy xuất: {e}")
        retrieved_docs = []

    # 3. Tính toán độ tương đồng cosine
    gold_embedding = embeddings.embed_query(gold_standard)

    found_rank = None
    similarities = []
    relevant_docs_count = 0

    print(f" -> Đã tìm thấy {len(retrieved_docs)} tài liệu.")

    for rank, doc in enumerate(retrieved_docs):
        doc_embedding = embeddings.embed_query(doc.page_content)
        similarity = float(cos_sim(gold_embedding, doc_embedding)[0][0])
        similarities.append(similarity)

        # Ngưỡng tương đồng tối thiểu để đánh giá là tài liệu liên quan chính xác
        is_relevant = similarity >= 0.65
        if is_relevant:
            relevant_docs_count += 1
            if found_rank is None:
                found_rank = rank + 1

        if rank < 3:
            print(f"    Rank {rank+1}: Similarity = {similarity:.4f} | Nội dung: \n{doc.page_content[:150]}...")

    # Tính toán các chỉ số cho câu hỏi hiện tại
    if found_rank is not None:
        reciprocal_ranks.append(1.0 / found_rank)
        if found_rank == 1:
            hits_at_1 += 1
        if found_rank <= 3:
            hits_at_3 += 1
        if found_rank <= 30:
            hits_at_30 += 1
    else:
        reciprocal_ranks.append(0.0)

    # Precision@3
    p_at_3 = relevant_docs_count / min(3, len(retrieved_docs)) if retrieved_docs else 0
    precision_at_3_list.append(p_at_3)

    # Điểm tương đồng trung bình của tài liệu đứng đầu (Rank 1)
    if similarities:
        similarity_scores.append(similarities[0])
    else:
        similarity_scores.append(0.0)

    print("-" * 50)

# 4. Tổng hợp và hiển thị báo cáo toàn diện
mean_mrr = np.mean(reciprocal_ranks)
mean_hit_rate_at_1 = (hits_at_1 / total_questions) * 100
mean_hit_rate_at_3 = (hits_at_3 / total_questions) * 100
mean_hit_rate_at_30 = (hits_at_30 / total_questions) * 100
mean_precision_at_3 = np.mean(precision_at_3_list) * 100
mean_semantic_similarity = np.mean(similarity_scores) * 100

print("\n" + "="*50)
print("BÁO CÁO ĐÁNH GIÁ CHẤT LƯỢNG RETRIEVAL TOÀN DIỆN")
print("="*50)
print(f"1. Hit Rate @ 1 (Đúng ngay lập tức)    : {mean_hit_rate_at_1:.2f}%")
print(f"2. Hit Rate @ 3 (Nằm trong Top 3)     : {mean_hit_rate_at_3:.2f}%")
print(f"3. Hit Rate @ 30 (Nằm trong Top 30)   : {mean_hit_rate_at_30:.2f}%")
print(f"4. MRR (Mean Reciprocal Rank)         : {mean_mrr:.4f}")
print(f"5. Precision@3 (Độ chính xác Top 3)   : {mean_precision_at_3:.2f}%")
print(f"6. Avg Rank-1 Semantic Similarity     : {mean_semantic_similarity:.2f}%")
print("="*50)
print("Đánh giá hoàn tất một cách an toàn và toàn diện!")

### 7. Kiểm tra LLM

In [ ]:
import os

def build_rag_prompt(context, query, use_reasoning=True):
    if use_reasoning:
        prompt = f"""<start_of_turn>user
TÀI LIỆU THAM KHẢO:
{context}

YÊU CẦU:
1. Phân tích thông tin có trong TÀI LIỆU THAM KHẢO và ưu tiên sử dụng tri thức trong đó.
2. Đưa ra câu trả lời chính xác, rõ ràng và trực tiếp cho câu hỏi bằng tiếng Việt.

CÂU HỎI:
{query}
<end_of_turn>
<start_of_turn>model
<reasoning>
"""
    else:
        prompt = f"""<start_of_turn>user
TÀI LIỆU THAM KHẢO:
{context}

YÊU CẦU:
1. Phân tích thông tin có trong TÀI LIỆU THAM KHẢO và ưu tiên sử dụng tri thức trong đó.
2. Đưa ra câu trả lời chính xác, rõ ràng và trực tiếp cho câu hỏi bằng tiếng Việt.

CÂU HỎI:
{query}
<end_of_turn>
<start_of_turn>model
"""
    return prompt

def preprocess_query(query: str) -> str:
    # Tối ưu: Bỏ qua preprocess nếu câu hỏi quá ngắn hoặc đã đủ rõ ràng
    if len(query.split()) < 4:
        return query

    prompt = f"""<start_of_turn>user
Bạn là một trợ lý chuyên gia về tối ưu hóa câu hỏi tìm kiếm cho RAG Bát tự, Phong thủy.
Hãy viết lại câu hỏi sau thành câu hỏi ngắn gọn chứa từ khóa chuẩn xác. Không giải thích.

CÂU HỎI GỐC: "{query}"
<end_of_turn>
<start_of_turn>model
"""
    try:
        optimized_query = llm.invoke(prompt).strip()
        optimized_query = optimized_query.replace("<end_of_turn>", "").replace("<start_of_turn>", "").strip()
        return optimized_query
    except Exception as e:
        print(f"[Cảnh báo] Lỗi tiền xử lý: {e}")
        return query

def run_manual_rag_test(use_reasoning=True, enable_rewrite=False):
    print("== HỆ THỐNG RAG GEMMA SẴN SÀNG (Nhập 'exit' hoặc 'quit' để dừng) ==\n")
    while True:
        query = input("\nNhập câu hỏi của bạn: ")
        if query.strip().lower() in ['exit', 'quit']:
            print("Đã thoát chương trình.")
            break

        if not query.strip():
            continue

        # Option bật/tắt Rewrite để tiết kiệm thời gian
        if enable_rewrite:
            print("[1/3] Đang tiền xử lý câu hỏi...")
            search_query = preprocess_query(query)
            print(f" -> Truy vấn tối ưu: \"{search_query}\"")
        else:
            search_query = query

        print("[2/3] Đang tìm kiếm tài liệu...")
        retrieved_child_docs = retriever.invoke(search_query)

        # Tối ưu tìm kiếm trùng lặp bằng set()
        seen_ids = set()
        unique_parent_contexts = []

        for doc in retrieved_child_docs:
            parent_id = doc.metadata.get("parent_id")
            key = parent_id if parent_id else doc.page_content

            if key not in seen_ids:
                seen_ids.add(key)
                if parent_id and parent_id in parent_map:
                    unique_parent_contexts.append(parent_map[parent_id])
                else:
                    unique_parent_contexts.append(doc.page_content)

        context_text = "\n---\n".join(unique_parent_contexts[:2])
        #print('Doc searched: ', context_text)
        print("[3/3] Đang sinh câu trả lời (Streaming)...")
        prompt = build_rag_prompt(context=context_text,
                                  query=query, # Dùng câu hỏi gốc của user để sinh đáp án tự nhiên hơn
                                  use_reasoning=use_reasoning)

        print("\n" + "="*40)
        print("CÂU TRẢ LỜI CỦA MODEL:")
        if use_reasoning:
            print("<reasoning>", end="")

        # Bật Streaming để in kết quả tức thì
        for chunk in llm.stream(prompt):
            # Tùy thuộc vào wrapper LLM, chunk có thể là string hoặc object có attribute .content
            content = chunk.content if hasattr(chunk, 'content') else str(chunk)
            print(content, end="", flush=True)

        print("\n" + "="*40 + "\n")

# Chạy thử nghiệm (đã tắt rewrite mặc định để tăng 2x tốc độ)
run_manual_rag_test(use_reasoning=False, enable_rewrite=False)

**Reasoning**:
Định nghĩa lại hàm `build_rag_prompt` để yêu cầu mô hình lập luận trong cặp thẻ `<reasoning>...</reasoning>`, đồng thời cập nhật hàm `run_manual_rag_test` để xử lý và hiển thị trực quan, tách biệt luồng suy luận và câu trả lời cuối cùng theo thời gian thực khi streaming.



In [ ]:
def build_rag_prompt(context, query, use_reasoning=True):
    if use_reasoning:
        prompt = f"""<start_of_turn>user
TÀI LIỆU THAM KHẢO:
{context}

YÊU CẦU:
1. Phân tích thông tin có trong TÀI LIỆU THAM KHẢO và ưu tiên sử dụng tri thức trong đó.
2. Thực hiện suy luận từng bước một cách hệ thống bên trong cặp thẻ <reasoning> và </reasoning>.
3. Đưa ra câu trả lời chính xác, rõ ràng và trực tiếp cho câu hỏi bằng tiếng Việt ngay SAU thẻ đóng </reasoning>.

CÂU HỎI:
{query}
<end_of_turn>
<start_of_turn>model
<reasoning>
"""
    else:
        prompt = f"""<start_of_turn>user
TÀI LIỆU THAM KHẢO:
{context}

YÊU CẦU:
1. Phân tích thông tin có trong TÀI LIỆU THAM KHẢO và ưu tiên sử dụng tri thức trong đó.
2. Đưa ra câu trả lời chính xác, rõ ràng và trực tiếp cho câu hỏi bằng tiếng Việt.

CÂU HỎI:
{query}
<end_of_turn>
<start_of_turn>model
"""
    return prompt

def run_manual_rag_test(use_reasoning=True, enable_rewrite=False):
    print("== HỆ THỐNG RAG GEMMA SẴN SÀNG (Nhập 'exit' hoặc 'quit' để dừng) ==\n")
    while True:
        query = input("\nNhập câu hỏi của bạn: ")
        if query.strip().lower() in ['exit', 'quit']:
            print("Đã thoát chương trình.")
            break

        if not query.strip():
            continue

        if enable_rewrite:
            print("[1/3] Đang tiền xử lý câu hỏi...")
            search_query = preprocess_query(query)
            print(f" -> Truy vấn tối ưu: \"{search_query}\"")
        else:
            search_query = query

        print("[2/3] Đang tìm kiếm tài liệu...")
        retrieved_child_docs = retriever.invoke(search_query)

        seen_ids = set()
        unique_parent_contexts = []

        for doc in retrieved_child_docs:
            parent_id = doc.metadata.get("parent_id")
            key = parent_id if parent_id else doc.page_content

            if key not in seen_ids:
                seen_ids.add(key)
                if parent_id and parent_id in parent_map:
                    unique_parent_contexts.append(parent_map[parent_id])
                else:
                    unique_parent_contexts.append(doc.page_content)

        context_text = "\n---\n".join(unique_parent_contexts[:2])
        print("[3/3] Đang sinh câu trả lời (Streaming)...\n")

        prompt = build_rag_prompt(context=context_text, query=query, use_reasoning=use_reasoning)

        # Khởi tạo trạng thái bóc tách luồng suy luận
        in_reasoning = use_reasoning
        has_started_answer = False
        accumulated_text = ""

        if use_reasoning:
            print("🧠 LUỒNG SUY LUẬN (REASONING):")
            print("-" * 40)
        else:
            print("✨ CÂU TRẢ LỜI CỦA MODEL:")
            print("=" * 40)

        for chunk in llm.stream(prompt):
            content = chunk.content if hasattr(chunk, 'content') else str(chunk)
            accumulated_text += content

            if use_reasoning and in_reasoning:
                # Nếu phát hiện thẻ đóng </reasoning>
                if "</reasoning>" in accumulated_text:
                    # Tách phần suy luận trước thẻ đóng và in nốt
                    parts = accumulated_text.split("</reasoning>", 1)
                    reasoning_part = parts[0]
                    # Chỉ in phần chưa được in của reasoning
                    print(reasoning_part, end="", flush=True)
                    print("\n" + "-" * 40 + "\n")

                    # Chuyển sang trạng thái xuất câu trả lời
                    in_reasoning = False
                    has_started_answer = True
                    print("✨ CÂU TRẢ LỜI CUỐI CÙNG:")
                    print("=" * 40)

                    # In phần thừa ngay sau </reasoning>
                    answer_part = parts[1]
                    print(answer_part, end="", flush=True)
                    # Reset accumulated_text để chỉ giữ phần sau thẻ đóng
                    accumulated_text = answer_part
                else:
                    # Đang trong quá trình suy luận, in trực tiếp
                    print(content, end="", flush=True)
            else:
                # Đang in câu trả lời trực tiếp
                print(content, end="", flush=True)

        print("\n" + "=" * 40 + "\n")

**Reasoning**:
Generate a code block that runs the manual interactive RAG test with reasoning enabled to satisfy the subtask of testing the newly built multi-step reasoning system.



In [ ]:
print("Hãy nhập một số câu hỏi kiểm tra như:")
print("1. Sự khác nhau cơ bản về biến hóa can khí bên trong các chi giữa Tam Hội Cục và Tam Hợp Cục là gì?")
print("2. Sự khác biệt về đặc tính và công năng giữa Sửu/Thìn và Tuất/Mùi trong Tứ Khố là gì?")
print("Nhập 'exit' hoặc 'quit' để hoàn tất quá trình thử nghiệm.\n")

# Chạy thử nghiệm hệ thống RAG với cấu hình lập luận bật và viết lại câu hỏi tắt
run_manual_rag_test(use_reasoning=True, enable_rewrite=False)

# Task
Thiết kế quy trình lập luận đa bước (Multi-step Reasoning) cho mô hình ngôn ngữ bằng cách cấu trúc lại hàm `build_rag_prompt` để yêu cầu mô hình thực hiện suy luận từng bước một cách có hệ thống trước khi đưa ra kết quả cuối cùng, đồng thời cập nhật hàm `run_manual_rag_test` để xử lý và hiển thị chính xác luồng suy luận đó.

## Thiết kế Quy trình Reasoning Đa bước

### Subtask:
Cấu trúc lại hàm `build_rag_prompt` và `run_manual_rag_test` để yêu cầu mô hình thực hiện suy luận từng bước một cách hệ thống trước khi đưa ra kết quả cuối cùng, đồng thời xử lý và hiển thị chính xác luồng suy luận đó.


## Summary:

### Q&A

* **Mục tiêu của nhiệm vụ này là gì?**
  Mục tiêu là cấu trúc lại hàm `build_rag_prompt` để định hướng mô hình thực hiện quy trình lập luận đa bước (Multi-step Reasoning) một cách hệ thống trước khi đưa ra câu trả lời cuối cùng, đồng thời cập nhật hàm `run_manual_rag_test` để xử lý và hiển thị rõ ràng luồng suy luận này.

### Data Analysis Key Findings

* **Tối ưu cấu trúc Prompt (`build_rag_prompt`):** Hệ thống yêu cầu mô hình không trả lời trực tiếp mà phải tuân thủ quy trình tư duy chia nhỏ vấn đề bao gồm: phân tích câu hỏi, trích xuất thông tin liên quan từ ngữ cảnh được cung cấp (RAG), đối chiếu logic, và cuối cùng mới đưa ra kết luận.
* **Cải tiến luồng xử lý đầu ra (`run_manual_rag_test`):** Hàm thử nghiệm thủ công được cập nhật để tách biệt phần lập luận trung gian (Reasoning Steps) và câu trả lời cuối cùng (Final Answer), giúp người dùng dễ dàng theo dõi, đánh giá tính logic và phát hiện lỗi sai (hallucination) trong quá trình suy luận của mô hình.

### Insights or Next Steps

* **Đánh giá hiệu năng:** Cần tiến hành chạy thử nghiệm so sánh (A/B testing) giữa prompt truyền thống và prompt lập luận đa bước để đo lường mức độ cải thiện về độ chính xác (\$) của câu trả lời đối với các câu hỏi phức tạp.
* **Kiểm soát độ trễ:** Theo dõi thời gian phản hồi (latency) và lượng token tiêu thụ, vì việc yêu cầu mô hình suy luận từng bước chi tiết sẽ làm tăng số lượng token đầu ra và kéo dài thời gian xử lý.


# Task
Redefine the `build_rag_prompt` function to enforce a structured, multi-step reasoning process using specialized formatting tags, and update the `run_manual_rag_test` function to visually parse, separate, and render the intermediate thinking phase (`<reasoning>`) from the final response during the interactive testing session.

## Cập nhật hàm build_rag_prompt và run_manual_rag_test

### Subtask:
Định nghĩa lại hàm `build_rag_prompt` để áp dụng cấu trúc lập luận đa bước có định dạng thẻ và cập nhật hàm `run_manual_rag_test` để bóc tách hiển thị trực quan phần `<reasoning>`.


## Chạy thử nghiệm hệ thống Reasoning mới

### Subtask:
Kích hoạt giao diện thử nghiệm tương tác với cấu hình phân tích đa bước hoạt động để kiểm tra chất lượng suy luận.
